# Multi-stream F1 per ensemble size (interactive)

Reads the per-trial CSVs and the held-out eval CSVs produced by
`optimization/synthetic_f1_multistream_optimize_optuna.py` and plots:

- every completed Optuna trial as one point at its ensemble size `N`,
  with the **train macro F1** on the y axis;
- the held-out **eval macro F1** of each best trial as a separate
  trace, so we can see overfitting (train high, eval lower);
- a per-stream F1 breakdown for the best trial at every size;
- the train-vs-eval gap as an explicit overfitting diagnostic.

Source layout (default `--output-dir synthetic_multistream_results`):

```
synthetic_multistream_results/
  <generator>/
    synthF1ms_<generator>_N<size>_S<n_streams>[_w<worker>].csv   # per-trial
    synthF1ms_<generator>_N<size>_S<n_streams>_eval.csv          # best-trial held-out eval
```

The Optuna writer can emit rows wider than the on-disk header (later
trials introduce new conditional params), so we parse these CSVs
tolerantly.


In [ ]:
import os
import re
import csv
import ast
import glob
import html

import numpy as np
import pandas as pd
import plotly.graph_objects as go

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
RESULTS_DIR = os.path.join(REPO_ROOT, 'synthetic_multistream_results')
print('REPO_ROOT  =', REPO_ROOT)
print('RESULTS    =', RESULTS_DIR)

TRIAL_FNAME_RE = re.compile(
    r'^synthF1ms_(?P<generator>.+)_N(?P<size>\d+)_S(?P<n_streams>\d+)'
    r'(?:_w(?P<worker>\d+))?\.csv$')
EVAL_FNAME_RE = re.compile(
    r'^synthF1ms_(?P<generator>.+)_N(?P<size>\d+)_S(?P<n_streams>\d+)_eval\.csv$')

HOVER_EXCLUDE = {
    'trial_id', 'macro_f1', 
    'per_stream_f1',  'per_stream_mean_delay', 
    'per_stream_tp', 'per_stream_fp',
    'train_indices', 'eval_indices', 'n_train_streams', 'error',
    
    'size', 'generator', 'n_streams', 'method', 'source_file', 'worker',
}

In [ ]:
def _read_ragged_csv(path: str) -> pd.DataFrame:
    """Read a possibly-ragged CSV (Optuna writer can append columns)."""
    with open(path, 'r', newline='') as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            return pd.DataFrame()
        rows = list(reader)
    if not rows:
        return pd.DataFrame(columns=header)
    max_w = max((len(r) for r in rows), default=len(header))
    if max_w > len(header):
        header = list(header) + [f'extra_{i}' for i in range(max_w - len(header))]
    padded = [r + [''] * (len(header) - len(r)) for r in rows]
    return pd.DataFrame(padded, columns=header)


def _parse_list(v):
    if v in ('', None):
        return None
    if isinstance(v, (list, tuple)):
        return list(v)
    try:
        out = ast.literal_eval(v)
        if isinstance(out, (list, tuple)):
            return [x for x in out]
    except (ValueError, SyntaxError):
        pass
    return None


LIST_COLS = [
    'per_stream_f1',  'per_stream_mean_delay', 
    'per_stream_tp', 'per_stream_fp',
    'train_indices', 'eval_indices',
    'train_per_f1',  'train_per_mean_delay', 
    'train_per_tp', 'train_per_fp',
    'eval_per_f1',  'eval_per_mean_delay', 
    'eval_per_tp', 'eval_per_fp',
]


def load_trials() -> pd.DataFrame:
    if not os.path.isdir(RESULTS_DIR):
        print(f'[skip] {RESULTS_DIR} not found')
        return pd.DataFrame()
    paths = sorted(glob.glob(os.path.join(RESULTS_DIR, '*', '*.csv'))
                   + glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
    frames = []
    for path in paths:
        name = os.path.basename(path)
        if EVAL_FNAME_RE.match(name):
            continue
        m = TRIAL_FNAME_RE.match(name)
        if not m:
            continue
        try:
            df = _read_ragged_csv(path)
        except Exception as e:
            print(f'  [skip] {path}: {e!r}')
            continue
        if df.empty or 'macro_f1' not in df.columns:
            continue
        df['macro_f1'] = pd.to_numeric(df['macro_f1'], errors='coerce')
        df = df.dropna(subset=['macro_f1']).copy()
        if df.empty:
            continue
        for col in LIST_COLS:
            if col in df.columns:
                df[col] = df[col].map(_parse_list)
        df['generator'] = m.group('generator')
        df['size'] = int(m.group('size'))
        df['n_streams'] = int(m.group('n_streams'))
        df['worker'] = int(m.group('worker')) if m.group('worker') else -1
        df['source_file'] = os.path.relpath(path, REPO_ROOT)
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)


def load_evals() -> pd.DataFrame:
    if not os.path.isdir(RESULTS_DIR):
        return pd.DataFrame()
    paths = sorted(glob.glob(os.path.join(RESULTS_DIR, '*', '*_eval.csv')))
    frames = []
    for path in paths:
        m = EVAL_FNAME_RE.match(os.path.basename(path))
        if not m:
            continue
        df = _read_ragged_csv(path)
        if df.empty:
            continue
        for col in LIST_COLS:
            if col in df.columns:
                df[col] = df[col].map(_parse_list)
        for col in ('train_macro_f1', 'eval_macro_f1',
                    ):
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        if 'size' in df.columns:
            df['size'] = pd.to_numeric(df['size'], errors='coerce').astype('Int64')
        else:
            df['size'] = int(m.group('size'))
        if 'generator' not in df.columns:
            df['generator'] = m.group('generator')
        df['n_streams'] = int(m.group('n_streams'))
        df['source_file'] = os.path.relpath(path, REPO_ROOT)
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)


trials = load_trials()
evals = load_evals()
if not trials.empty:
    print(f"Loaded {len(trials)} trial rows / generators={sorted(trials['generator'].unique())} / sizes={sorted(trials['size'].unique())}")
else:
    print('No trial rows loaded.')
if not evals.empty:
    print(f"Loaded {len(evals)} eval rows / generators={sorted(evals['generator'].unique())} / sizes={sorted(evals['size'].dropna().astype(int).unique())}")
else:
    print('No eval rows loaded yet.')
trials.head()

In [ ]:
def _format_config_html(row: pd.Series) -> str:
    parts = []
    head_bits = []
    tid = row.get('trial_id', '')
    if tid not in ('', None) and not (isinstance(tid, float) and np.isnan(tid)):
        head_bits.append(f'trial {tid}')
    head_bits.append(f"macroF1={float(row['macro_f1']):.4f}")
    mdr = row.get( '')
    if mdr not in ('', None):
        try:
            head_bits.append(f'macroF1_recall={float(mdr):.3f}')
        except (TypeError, ValueError):
            pass
    parts.append('<b>' + html.escape(' | '.join(head_bits)) + '</b>')

    for col, label in (('per_stream_f1', 'per_stream_f1'),
                       ('per_stream_tp', 'per_stream_tp'),
                       ('per_stream_fp', 'per_stream_fp')):
        v = row.get(col)
        if isinstance(v, list) and v:
            if all(isinstance(x, (int, np.integer)) for x in v):
                s = ', '.join(str(int(x)) for x in v)
            else:
                s = ', '.join(f'{float(x):.3f}' for x in v)
            parts.append(f"<span style='color:#555'>{label}</span> = [" + html.escape(s) + ']')

    err = row.get('error', '')
    if isinstance(err, str) and err:
        parts.append('<span style="color:#b22">error: ' + html.escape(err) + '</span>')

    cfg_items = []
    for k, v in row.items():
        if k in HOVER_EXCLUDE:
            continue
        if isinstance(v, list):
            continue
        if v in ('', None):
            continue
        if isinstance(v, float) and np.isnan(v):
            continue
        if isinstance(v, float):
            vs = f'{v:.5g}'
        else:
            vs = str(v)
        cfg_items.append((str(k), vs))

    def _sort_key(item):
        k = item[0]
        m = re.match(r'slot(\d+)_(.*)', k)
        if m:
            return (0, int(m.group(1)), 0 if m.group(2) == 'type' else 1, m.group(2))
        return (1, 0, 0, k)
    cfg_items.sort(key=_sort_key)
    if cfg_items:
        parts.append('<br>'.join(
            f"<span style='color:#555'>{html.escape(k)}</span> = {html.escape(v)}"
            for k, v in cfg_items))
    parts.append('<span style="color:#888;font-size:10px">' +
                 html.escape(str(row.get('source_file', ''))) + '</span>')
    return '<br>'.join(parts)


JITTER = 0.08


def plot_group(df_g: pd.DataFrame, df_eval: pd.DataFrame, title: str):
    sizes_sorted = sorted(df_g['size'].unique())
    rng = np.random.default_rng(0)
    x = df_g['size'].to_numpy(dtype=float)
    x_jit = x * np.exp(rng.uniform(-JITTER, JITTER, size=x.shape))
    hover = [_format_config_html(r) for _, r in df_g.iterrows()]
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=x_jit, y=df_g['macro_f1'].to_numpy(),
        mode='markers',
        name='trial (train macro F1)',
        marker=dict(color='#1f77b4', symbol='circle', size=7,
                    opacity=0.5, line=dict(width=0)),
        text=hover,
        hovertemplate='%{text}<extra></extra>',
    ))
    best = df_g.groupby('size')['macro_f1'].max().sort_index()
    fig.add_trace(go.Scatter(
        x=best.index.to_numpy(dtype=float), y=best.values,
        mode='lines+markers',
        name='best train macro F1',
        line=dict(color='#1f77b4', width=2),
        marker=dict(color='#1f77b4', symbol='circle', size=12,
                    line=dict(width=1.5, color='#1f77b4')),
        hoverinfo='skip',
    ))
    if df_eval is not None and not df_eval.empty:
        df_eval = df_eval.dropna(subset=['size', 'eval_macro_f1']).copy()
        df_eval['size'] = df_eval['size'].astype(int)
        df_eval = df_eval.sort_values('size')
        if 'train_macro_f1' in df_eval.columns:
            df_eval = (df_eval.sort_values('train_macro_f1', ascending=False)
                              .drop_duplicates(subset=['size']))
        fig.add_trace(go.Scatter(
            x=df_eval['size'].to_numpy(dtype=float),
            y=df_eval['eval_macro_f1'].to_numpy(),
            mode='lines+markers',
            name='best on HELD-OUT eval',
            line=dict(color='#d62728', width=2, dash='dot'),
            marker=dict(color='#d62728', symbol='diamond', size=12,
                        line=dict(width=1.5, color='#d62728')),
            hovertemplate='N=%{x}  eval macro F1=%{y:.4f}<extra></extra>',
        ))
        if 'train_macro_f1' in df_eval.columns:
            fig.add_trace(go.Scatter(
                x=df_eval['size'].to_numpy(dtype=float),
                y=df_eval['train_macro_f1'].to_numpy(),
                mode='lines+markers',
                name='best on TRAIN (recomputed)',
                line=dict(color='#2ca02c', width=2, dash='dot'),
                marker=dict(color='#2ca02c', symbol='square', size=10,
                            line=dict(width=1.5, color='#2ca02c')),
                hovertemplate='N=%{x}  train macro F1=%{y:.4f}<extra></extra>',
            ))
    fig.update_layout(
        title=f'macro F1 per trial vs. ensemble size  -  {title}',
        xaxis=dict(title='Ensemble size N', type='log',
                   tickvals=sizes_sorted,
                   ticktext=[str(s) for s in sizes_sorted]),
        yaxis=dict(title='macro F1'),
        hovermode='closest',
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1.0),
        template='plotly_white',
        height=540,
    )
    return fig


if trials.empty:
    print('No data found yet. Re-run once the optimization jobs have produced CSVs.')
else:
    for (generator, n_streams), df_g in trials.groupby(['generator', 'n_streams']):
        if not evals.empty:
            df_e = evals[(evals['generator'] == generator)
                         & (evals['n_streams'] == n_streams)]
        else:
            df_e = pd.DataFrame()
        plot_group(df_g, df_e, f'{generator}  (S={n_streams})').show()

In [ ]:
if not trials.empty:
    summary = (trials.groupby(['generator', 'n_streams', 'size'])
                     .agg(n_trials=('macro_f1', 'count'),
                          best_train_f1=('macro_f1', 'max'),
                          mean_train_f1=('macro_f1', 'mean'))
                     .reset_index()
                     .sort_values(['generator', 'n_streams', 'size']))
    if not evals.empty:
        eval_keep = evals.dropna(subset=['size']).copy()
        eval_keep['size'] = eval_keep['size'].astype(int)
        eval_keep = (eval_keep.sort_values('train_macro_f1', ascending=False)
                              .drop_duplicates(subset=['generator', 'n_streams', 'size'])
                              [['generator', 'n_streams', 'size',
                                'train_macro_f1', 'eval_macro_f1',
                                ]])
        summary = summary.merge(eval_keep,
                                on=['generator', 'n_streams', 'size'],
                                how='left')
    display(summary)

## Per-stream F1 for the best trial at each size

For each `(generator, n_streams, size)` we take the trial with the highest
*train* macro F1 and show its per-stream F1 vector as a small bar chart.
Stream indices come from the `train_indices` column. If a held-out eval
row exists for the same `(generator, size)` we also overlay the per-stream
F1 on the eval streams (using `eval_indices`).


In [ ]:
def _has_list(v):
    return isinstance(v, list) and len(v) > 0


if not trials.empty and 'per_stream_f1' in trials.columns:
    rows = trials[trials['per_stream_f1'].map(_has_list)].copy()
    if rows.empty:
        print('No rows with per_stream_f1 available.')
    else:
        for (generator, n_streams), df_g in rows.groupby(['generator', 'n_streams']):
            best_idx = df_g.groupby('size')['macro_f1'].idxmax().sort_index()
            best_rows = df_g.loc[best_idx].sort_values('size')
            fig = go.Figure()
            for _, r in best_rows.iterrows():
                psf = r['per_stream_f1']
                idx = r.get('train_indices')
                if isinstance(idx, list) and len(idx) == len(psf):
                    xs = [int(x) for x in idx]
                else:
                    xs = list(range(len(psf)))
                fig.add_trace(go.Bar(
                    x=xs, y=psf,
                    name=f"N={int(r['size'])} train (macroF1={r['macro_f1']:.3f})",
                ))
            if not evals.empty:
                df_e = evals[(evals['generator'] == generator)
                             & (evals['n_streams'] == n_streams)].copy()
                if not df_e.empty and 'eval_per_f1' in df_e.columns:
                    df_e = df_e.dropna(subset=['size'])
                    df_e['size'] = df_e['size'].astype(int)
                    df_e = (df_e.sort_values('train_macro_f1', ascending=False)
                                  .drop_duplicates(subset=['size'])
                                  .sort_values('size'))
                    for _, r in df_e.iterrows():
                        psf = r['eval_per_f1']
                        idx = r.get('eval_indices')
                        if not isinstance(psf, list):
                            continue
                        if isinstance(idx, list) and len(idx) == len(psf):
                            xs = [int(x) for x in idx]
                        else:
                            xs = list(range(len(psf)))
                        fig.add_trace(go.Bar(
                            x=xs, y=psf,
                            name=f"N={int(r['size'])} eval (macroF1={float(r['eval_macro_f1']):.3f})",
                            marker_pattern_shape='/',
                        ))
            fig.update_layout(
                title=f'Per-stream F1 for best trial per size  -  {generator}  (S={n_streams})',
                xaxis=dict(title='Stream index (in full N-stream list)'),
                yaxis=dict(title='F1'),
                barmode='group',
                template='plotly_white',
                height=480,
            )
            fig.show()

## Train vs eval gap

If the best trial generalises, train and eval macro F1 should be close. A
large gap (train >> eval) means the optimizer overfit to the train seeds.


In [ ]:
if evals.empty:
    print('No eval CSVs yet.')
else:
    df = evals.dropna(subset=['size']).copy()
    df['size'] = df['size'].astype(int)
    df = (df.sort_values('train_macro_f1', ascending=False)
            .drop_duplicates(subset=['generator', 'n_streams', 'size'])
            .sort_values(['generator', 'n_streams', 'size']))
    df['gap'] = df['train_macro_f1'] - df['eval_macro_f1']
    fig = go.Figure()
    for (gen, ns), df_g in df.groupby(['generator', 'n_streams']):
        fig.add_trace(go.Scatter(
            x=df_g['size'].to_numpy(),
            y=df_g['gap'].to_numpy(),
            mode='lines+markers',
            name=f'{gen}  (S={ns})',
            hovertemplate=('N=%{x}  gap=%{y:.4f}<br>'
                           'train=%{customdata[0]:.4f}  eval=%{customdata[1]:.4f}'
                           '<extra></extra>'),
            customdata=np.stack([df_g['train_macro_f1'].to_numpy(),
                                 df_g['eval_macro_f1'].to_numpy()], axis=1),
        ))
    fig.update_layout(
        title='Train minus eval macro F1 (overfitting gap)',
        xaxis=dict(title='Ensemble size N', type='log'),
        yaxis=dict(title='train macro F1  -  eval macro F1'),
        template='plotly_white',
        height=420,
    )
    fig.add_hline(y=0, line=dict(color='#888', dash='dash'))
    fig.show()
    display(df[['generator', 'n_streams', 'size',
                'train_macro_f1', 'eval_macro_f1', 'gap',
                ]])

## Greedy ensemble from N=1 pool (overlay)

Loads CSVs produced by `optimization/greedy_ensemble_from_pool.py` (one row per accepted greedy step) and overlays the greedy train+eval macro F1 curves on top of the joint-search results above. File name convention assumed: `greedy_<generator>_S<n_streams>.csv` placed in `RESULTS_DIR` or anywhere under it.


In [ ]:
GREEDY_FNAME_RE = re.compile(r'^greedy_(?P<generator>.+)_S(?P<n_streams>\d+)\.csv$')

def load_greedy() -> pd.DataFrame:
    paths = []
    if os.path.isdir(RESULTS_DIR):
        paths += glob.glob(os.path.join(RESULTS_DIR, '**', 'greedy_*.csv'), recursive=True)
    paths += glob.glob(os.path.join(REPO_ROOT, 'greedy_*.csv'))
    paths = sorted(set(paths))
    frames = []
    for path in paths:
        m = GREEDY_FNAME_RE.match(os.path.basename(path))
        if not m:
            continue
        try:
            df = pd.read_csv(path)
        except Exception as e:
            print(f'  [skip] {path}: {e!r}')
            continue
        if df.empty:
            continue
        for col in ('train_per_stream_f1', 'eval_per_stream_f1',
                    'members_kinds', 'members_sources'):
            if col in df.columns:
                df[col] = df[col].map(_parse_list)
        for col in ('train_macro_f1', 'eval_macro_f1'):
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        df['n'] = pd.to_numeric(df.get('n', df.get('step')), errors='coerce').astype('Int64')
        df['generator'] = m.group('generator')
        df['n_streams'] = int(m.group('n_streams'))
        df['source_file'] = os.path.relpath(path, REPO_ROOT)
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)


greedy = load_greedy()
if greedy.empty:
    print('No greedy_*.csv files found. Run optimization/greedy_ensemble_from_pool.py first.')
else:
    print(f"Loaded {len(greedy)} greedy step rows / generators={sorted(greedy['generator'].unique())} / n_streams={sorted(greedy['n_streams'].unique())}")
    display(greedy[['generator','n_streams','n','added_kind','ens_crit','train_macro_f1','eval_macro_f1']].head(20))


In [ ]:
def plot_group_with_greedy(df_g, df_eval, df_greedy, title):
    fig = plot_group(df_g, df_eval, title)
    if df_greedy is not None and not df_greedy.empty:
        dg = df_greedy.dropna(subset=['n','train_macro_f1']).copy()
        dg['n'] = dg['n'].astype(int)
        dg = dg.sort_values('n')
        hover_g = []
        for _, r in dg.iterrows():
            mk = r.get('members_kinds')
            mk_str = ', '.join(mk) if isinstance(mk, list) else ''
            hover_g.append(
                f"N={int(r['n'])}  +{r.get('added_kind','?')}  ens_crit={r.get('ens_crit','?')}<br>"
                f"train={float(r['train_macro_f1']):.4f}  eval={float(r['eval_macro_f1']):.4f}<br>"
                f"members: {html.escape(mk_str)}"
            )
        fig.add_trace(go.Scatter(
            x=dg['n'].to_numpy(dtype=float),
            y=dg['train_macro_f1'].to_numpy(),
            mode='lines+markers',
            name='greedy train macro F1',
            line=dict(color='#ff7f0e', width=2),
            marker=dict(color='#ff7f0e', symbol='triangle-up', size=11,
                        line=dict(width=1.5, color='#ff7f0e')),
            text=hover_g,
            hovertemplate='%{text}<extra></extra>',
        ))
        if 'eval_macro_f1' in dg.columns:
            fig.add_trace(go.Scatter(
                x=dg['n'].to_numpy(dtype=float),
                y=dg['eval_macro_f1'].to_numpy(),
                mode='lines+markers',
                name='greedy eval macro F1',
                line=dict(color='#9467bd', width=2, dash='dot'),
                marker=dict(color='#9467bd', symbol='triangle-down', size=11,
                            line=dict(width=1.5, color='#9467bd')),
                text=hover_g,
                hovertemplate='%{text}<extra></extra>',
            ))
    return fig


if trials.empty:
    print('No trial rows; cannot overlay.')
else:
    for (generator, n_streams), df_g in trials.groupby(['generator', 'n_streams']):
        df_e = evals[(evals['generator'] == generator) & (evals['n_streams'] == n_streams)] if not evals.empty else pd.DataFrame()
        df_gr = greedy[(greedy['generator'] == generator) & (greedy['n_streams'] == n_streams)] if not greedy.empty else pd.DataFrame()
        plot_group_with_greedy(df_g, df_e, df_gr, f'{generator}  (S={n_streams})  + greedy').show()
